In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/hasib111/submittal-document/2970 SANTA ROSA AVE-CONSTRUCTION PLANS.pdf


In [2]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 61.1 MB/s eta 0:00:00


In [3]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 8.1 MB/s eta 0:00:00


# Another workFlow

In [4]:
"""
PDF Page-by-Page Processor with Claude API
==========================================
Workflow:
1. Split PDF into individual pages
2. For each page, try uploading as PDF to Claude Files API
3. If SUCCESS → keep that page PDF for final merge
4. If ERROR  → convert page to 300 DPI image → upload image to Claude →
               get summary → convert summary to PDF
5. Merge all pages (originals + summary PDFs) in order into one final PDF
6. Output to /kaggle/working/
"""

'\nPDF Page-by-Page Processor with Claude API\n==========================================\nWorkflow:\n1. Split PDF into individual pages\n2. For each page, try uploading as PDF to Claude Files API\n3. If SUCCESS → keep that page PDF for final merge\n4. If ERROR  → convert page to 300 DPI image → upload image to Claude →\n               get summary → convert summary to PDF\n5. Merge all pages (originals + summary PDFs) in order into one final PDF\n6. Output to /kaggle/working/\n'

In [5]:
!apt-get install -y poppler-utils -qq
!pip install -q pypdf reportlab pdf2image Pillow anthropic

Selecting previously unselected package poppler-utils.
(Reading database ... 120314 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.12_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.12) ...
Setting up poppler-utils (22.02.0-2ubuntu0.12) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 25.0 MB/s eta 0:00:00


In [6]:
import os
import sys
import time
import glob
from pypdf import PdfReader, PdfWriter
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_LEFT
from pdf2image import convert_from_path
from anthropic import Anthropic

In [7]:

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
anthropic_api_key = user_secrets.get_secret("ANTHROPIC_API_KEY")
FILES_API_BETA = "files-api-2025-04-14"
MODEL = "claude-sonnet-4-5"  # or "claude-sonnet-4-5-20250514"

INPUT_PDF = "/kaggle/input/datasets/hasib111/submittal-document/2970 SANTA ROSA AVE-CONSTRUCTION PLANS.pdf"
OUTPUT_DIR = "/kaggle/working"
FINAL_OUTPUT = os.path.join(OUTPUT_DIR, "final_combined_output.pdf")
TEMP_DIR = os.path.join(OUTPUT_DIR, "temp_pages")

DPI = 300  # For image conversion fallback

In [8]:
SUMMARY_PROMPT = """You are a construction plan review expert. This is a single page from a construction plan set.

Provide a detailed summary of this page including:
- Sheet number and title (if visible)
- Type of drawing (site plan, floor plan, elevation, detail, schedule, etc.)
- Key information shown (dimensions, notes, specifications, materials, etc.)
- Any code references or compliance notes
- Any significant details, callouts, or annotations

Be thorough but organized. Use clear headings."""

In [9]:
client = Anthropic(api_key=anthropic_api_key)

In [10]:
def upload_pdf(pdf_path):
    try:
        with open(pdf_path, 'rb') as file:
            file_content = file.read()

        filename = os.path.basename(pdf_path)

        file_response = client.beta.files.upload(
            file=(filename, file_content, "application/pdf"),
            betas=[FILES_API_BETA]
        )
        print(f"  Uploaded PDF: {filename} | ID: {file_response.id} | Size: {file_response.size_bytes} bytes")
        return file_response.id
    except Exception as e:
        print(f"  Error uploading PDF {pdf_path}: {e}")
        return None


def upload_image(image_path):
    try:
        with open(image_path, 'rb') as file:
            file_content = file.read()

        filename = os.path.basename(image_path)

        file_response = client.beta.files.upload(
            file=(filename, file_content, "image/png"),
            betas=[FILES_API_BETA]
        )
        print(f"  Uploaded Image: {filename} | ID: {file_response.id} | Size: {file_response.size_bytes} bytes")
        return file_response.id
    except Exception as e:
        print(f"  Error uploading image {image_path}: {e}")
        return None

def query_pdf_page(file_id, prompt):
    content = [
        {
            "type": "document",
            "source": {
                "type": "file",
                "file_id": file_id
            }
        },
        {
            "type": "text",
            "text": prompt
        }
    ]

    full_response = ""
    with client.beta.messages.stream(
        model=MODEL,
        max_tokens=16000,
        betas=[FILES_API_BETA],
        messages=[{"role": "user", "content": content}],
    ) as stream:
        for text in stream.text_stream:
            full_response += text

    return full_response

def query_image_page(file_id, prompt):
    content = [
        {
            "type": "image",
            "source": {
                "type": "file",
                "file_id": file_id
            }
        },
        {
            "type": "text",
            "text": prompt
        }
    ]

    full_response = ""
    with client.beta.messages.stream(
        model=MODEL,
        max_tokens=16000,
        betas=[FILES_API_BETA],
        messages=[{"role": "user", "content": content}],
    ) as stream:
        for text in stream.text_stream:
            full_response += text

    return full_response

def ensure_dirs():
    os.makedirs(TEMP_DIR, exist_ok=True)
    os.makedirs(OUTPUT_DIR, exist_ok=True)


def split_pdf_pages(input_pdf):
    """Split PDF into individual single-page PDFs."""
    reader = PdfReader(input_pdf)
    total = len(reader.pages)
    print(f"📄 Input PDF has {total} pages.")

    page_paths = []
    for i, page in enumerate(reader.pages):
        writer = PdfWriter()
        writer.add_page(page)
        out_path = os.path.join(TEMP_DIR, f"page_{i+1:03d}.pdf")
        with open(out_path, "wb") as f:
            writer.write(f)
        page_paths.append(out_path)

    return page_paths


def convert_page_to_image(pdf_path, page_num):
    img_path = os.path.join(TEMP_DIR, f"page_{page_num:03d}.png")
    images = convert_from_path(pdf_path, dpi=DPI, first_page=1, last_page=1)
    images[0].save(img_path, "PNG")
    print(f"  🖼️  Converted to image: {img_path} ({os.path.getsize(img_path)/1024:.0f} KB)")
    return img_path

def summary_to_pdf(summary_text, page_num):
    pdf_path = os.path.join(TEMP_DIR, f"summary_page_{page_num:03d}.pdf")

    doc = SimpleDocTemplate(
        pdf_path, pagesize=letter,
        leftMargin=0.75*inch, rightMargin=0.75*inch,
        topMargin=0.75*inch, bottomMargin=0.75*inch,
    )

    styles = getSampleStyleSheet()
    title_style = ParagraphStyle('PageTitle', parent=styles['Heading1'], fontSize=14, spaceAfter=12)
    body_style = ParagraphStyle('SummaryBody', parent=styles['Normal'], fontSize=10, leading=14, alignment=TA_LEFT, spaceAfter=6)

    story = []
    story.append(Paragraph(f"Page {page_num} — AI-Generated Summary", title_style))
    story.append(Spacer(1, 6))

    for para in summary_text.split('\n'):
        para = para.strip()
        if not para:
            story.append(Spacer(1, 6))
            continue
        para = para.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
        if para.startswith('#'):
            para = para.lstrip('#').strip()
            story.append(Paragraph(f"<b>{para}</b>", body_style))
        elif para.startswith('**') and para.endswith('**'):
            para = para.strip('*').strip()
            story.append(Paragraph(f"<b>{para}</b>", body_style))
        else:
            story.append(Paragraph(para, body_style))

    doc.build(story)
    print(f"  📄 Summary PDF created: {pdf_path}")
    return pdf_path


def merge_pdfs(pdf_paths, output_path):
    """Merge a list of PDF files into one."""
    writer = PdfWriter()
    for p in pdf_paths:
        reader = PdfReader(p)
        for page in reader.pages:
            writer.add_page(page)
    with open(output_path, "wb") as f:
        writer.write(f)
    print(f"\n✅ Final merged PDF: {output_path} ({os.path.getsize(output_path)/1024:.0f} KB)")

def main():
    ensure_dirs()

    print("=" * 70)
    print("PDF PAGE-BY-PAGE PROCESSOR WITH CLAUDE FILES API")
    print("=" * 70)

    # Step 1: Split PDF into pages
    print("\n📂 Step 1: Splitting PDF into individual pages...")
    page_pdf_paths = split_pdf_pages(INPUT_PDF)
    total_pages = len(page_pdf_paths)

    # Step 2: Process each page
    print(f"\n🔄 Step 2: Processing {total_pages} pages...\n")

    final_page_pdfs = []
    success_count = 0
    error_count = 0

    for i, page_pdf in enumerate(page_pdf_paths):
        page_num = i + 1
        print(f"─── Page {page_num}/{total_pages} ───")

        # Case 1: Try uploading as PDF (same pattern, just "application/pdf")
        file_id = upload_pdf(page_pdf)

        if file_id is not None:
            # Verify Claude can read it
            try:
                test_response = query_pdf_page(file_id, "Respond with OK if you can read this page.")
                print(f"  ✅ PDF page readable: {test_response.strip()[:50]}")
                final_page_pdfs.append(page_pdf)
                success_count += 1
            except Exception as e:
                print(f"  ❌ PDF page NOT readable: {e}")
                # Fall through to image path
                file_id = None

        if file_id is None:
            # Case 2: Convert to image → upload via Files API → get summary → make PDF
            error_count += 1
            print(f"  ⚠️  Falling back to image path...")

            # Convert to 300 DPI PNG
            img_path = convert_page_to_image(page_pdf, page_num)

            # Upload image (your exact upload_image function)
            img_file_id = upload_image(img_path)

            if img_file_id is not None:
                # Query with image (your exact pattern: type=image, source=file)
                summary = query_image_page(img_file_id, SUMMARY_PROMPT)
                print(f"  📝 Summary generated for page {page_num} ({len(summary)} chars)")
            else:
                summary = f"[Page {page_num}] — Error: Could not upload this page."

            # Convert summary to PDF
            summary_pdf = summary_to_pdf(summary, page_num)
            final_page_pdfs.append(summary_pdf)

        time.sleep(1)
        print()

    # Step 3: Merge all into final PDF
    print("=" * 70)
    print(f"📊 Results: {success_count} pages as original PDF, {error_count} pages as AI summaries")
    print(f"\n📎 Step 3: Merging {len(final_page_pdfs)} pages into final PDF...")

    merge_pdfs(final_page_pdfs, FINAL_OUTPUT)

    print(f"\n🎉 Done! Output saved to: {FINAL_OUTPUT}")
    print("=" * 70)


if __name__ == "__main__":
    main()

PDF PAGE-BY-PAGE PROCESSOR WITH CLAUDE FILES API

📂 Step 1: Splitting PDF into individual pages...
📄 Input PDF has 70 pages.

🔄 Step 2: Processing 70 pages...

─── Page 1/70 ───
  Uploaded PDF: page_001.pdf | ID: file_011CYcjgMawnWt5deKGxQktn | Size: 1598801 bytes
  ✅ PDF page readable: OK

I can read this page. It's a cover sheet (G001

─── Page 2/70 ───
  Uploaded PDF: page_002.pdf | ID: file_011CYcjgonHjECz4eczBfd9j | Size: 179182 bytes
  ✅ PDF page readable: OK

I can read this page. It appears to be an arch

─── Page 3/70 ───
  Uploaded PDF: page_003.pdf | ID: file_011CYcjhLLKaLczohZ27WT7B | Size: 34105475 bytes
  ✅ PDF page readable: OK

I can read this page. It appears to be a Calif

─── Page 4/70 ───
  Uploaded PDF: page_004.pdf | ID: file_011CYcjibWeaEq6xnUFT2FPZ | Size: 35930318 bytes
  ✅ PDF page readable: OK

I can read this page. It shows a California Gr

─── Page 5/70 ───
  Uploaded PDF: page_005.pdf | ID: file_011CYcjjt758CjDqa3xxk64Q | Size: 31456929 bytes
  ✅ PDF page 

In [11]:
file_ids = [
    "file_011CYc44XctYoaSryG4WF36o",
    "file_011CYc454Fe8WAwYXAh569KV",
    "file_011CYc45ph6NBQ4Hc59TUY3R",
    "file_011CYc46MQnRxAVMPEX3Z3Ly",
    "file_011CYc46oujKNTfJHGzRLfCT",
    "file_011CYc47EijQEwcwXJ5J9Rbs",
    "file_011CYc47cTBn9mPfDJZ61sP5",
    "file_011CYc48RVAchwYKjasrt8FR",
    "file_011CYc49427ba3goEzDeX4ME",
    "file_011CYc49hCW3wHxwnpbP7k1u",
    "file_011CYc4ASmN5QSNxrFDQ2ixU",
    "file_011CYc4AwTGnv5JETmFiFb5o",
    "file_011CYc4BPyTwY3pegivnSHii",
    "file_011CYc4C1ozRAhwX9eP9JboV",
    "file_011CYc4CndWwNxPS2HF1LnQo",
    "file_011CYc4DFpPGWaBjkiXBc2oP",
    "file_011CYc4DtFNqAZRHDUkxTrN4",
    "file_011CYc4EaQN1ZGC2HBjS1Ltj",
    "file_011CYc4FAyyrD2SmKxsJJSDw",
    "file_011CYc4FhkA12aEnkzRAjpt5",
    "file_011CYc4G5x9S6Cf4iTkVQDax",
    "file_011CYc4GWUmMhjfecbz8ReNB",
    "file_011CYc4H2s8pGJdSGiKbazGC",
    "file_011CYc4HYueDD1TwQMuuyuLA",
    "file_011CYc4HyjPzUTXQnBizEh4q",
    "file_011CYc4JQ4MwbpViABoaPnrQ",
    "file_011CYc4JsDzq54jbrTeu6TYq",
]

# Build content: all file docs + your prompt
content = [
    {"type": "document", "source": {"type": "file", "file_id": fid}}
    for fid in file_ids
]
content.append({"type": "text", "text": "Treats these files as one document, and tell me the scope of the work from the cover page"})

message = client.beta.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=4096,
    betas=["files-api-2025-04-14"],
    messages=[{"role": "user", "content": content}],
)

print(message.content[0].text)

Based on the cover sheet (A0) of this architectural drawing set, the **scope of work** is:

**FIRE RE-BUILD - NEW RESIDENCE**

This project involves the construction of a **new 2,996 square foot single family residence with 4 bedrooms and 3 bathrooms** at **3551 Parker Hill Court, Santa Rosa, CA 95404**.

Key project details from the cover sheet:
- **Total Floor Area**: 2,996 S.F.
- **First Floor**: 1,740 S.F. (including 504 S.F. garage and 1,236 S.F. living space)
- **Second Floor**: 1,256 S.F.
- **Lot Size**: 0.10 acres (4,356 S.F.)
- **Zoning**: R-1-6-RC
- **Construction Type**: VB
- **Occupancy Group**: R-3
- **Building Height**: 28'-1" +/-
- **Fire Sprinkler System**: Yes (automatic fire extinguishing system required)

The project is specifically noted as a "fire re-build," indicating this is a replacement residence being constructed after a fire destroyed the previous structure. The new home will comply with 2022 building codes and includes special Wildland Urban Interface (WUI) 